Part A1: Imports, constants, custom ModifiedLunarLanderWrapper, and helper functions.

In [13]:
# ==========================================================
# Assignment 2
# Task (a): Modified LunarLander Environment
# ==========================================================

import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd

# ----------------------------
# Global Parameters
# ----------------------------

ENGINE_FAILURE_PROB = 0.15
FUEL_PENALTY = 0.3
LANDING_BONUS = 50

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [14]:
#Modified Environment Wrapper
class ModifiedLunarLanderWrapper(gym.Wrapper):
    """
    Custom wrapper implementing stochastic engine failures.

    Modifications:
    1. 15% probability that actions {1,2,3}
       are replaced by action 0.

    2. Fuel penalty:
       Every attempted thruster action incurs
       a penalty of 0.3 regardless of whether
       the engine actually fires.

    3. Safe landing bonus:
       +50 reward for satisfying the safe landing
       conditions.
    """

    def __init__(
            self,
            env,
            failure_probability=0.15,
            fuel_penalty=0.3,
            landing_bonus=50):

        super().__init__(env)

        self.failure_probability = failure_probability
        self.fuel_penalty = fuel_penalty
        self.landing_bonus = landing_bonus

        # statistics (used only for verification)
        self.total_thruster_attempts = 0
        self.total_engine_failures = 0

    def reset(self, **kwargs):
        """
        Reset environment.
        """

        return self.env.reset(**kwargs)

    def step(self, action):
        """
        Executes one environment step with
        stochastic actuator failures.
        """

        ###################################################
        # STEP 1
        # Store original action
        ###################################################

        selected_action = action
        executed_action = selected_action

        ###################################################
        # STEP 2
        # Simulate engine failure
        ###################################################

        if selected_action in [1, 2, 3]:

            self.total_thruster_attempts += 1

            r = np.random.rand()

            if r < self.failure_probability:

                executed_action = 0

                self.total_engine_failures += 1

        ###################################################
        # STEP 3
        # Execute action
        ###################################################

        observation, base_reward, terminated, truncated, info = \
            self.env.step(executed_action)

        ###################################################
        # STEP 4
        # Modified reward
        ###################################################

        reward = base_reward

        # Fuel penalty depends on
        # ORIGINAL selected action

        if selected_action in [1, 2, 3]:

            reward -= self.fuel_penalty

        ###################################################
        # STEP 5
        # Safe landing bonus
        ###################################################

        safe_landing = (
            terminated
            and not truncated
            and observation[6] == 1
            and observation[7] == 1
            and abs(observation[2]) < 0.10
            and abs(observation[3]) < 0.10
            and abs(observation[4]) < 0.10
        )

        if safe_landing:

            reward += self.landing_bonus

        ###################################################
        # STEP 6
        # Return environment output
        ###################################################

        return (
            observation,
            reward,
            terminated,
            truncated,
            info
        )

    ###################################################
    # Helper function
    ###################################################

    def get_failure_rate(self):

        if self.total_thruster_attempts == 0:
            return 0

        return (
            self.total_engine_failures
            / self.total_thruster_attempts
        )

    ###################################################

    def reset_statistics(self):

        self.total_thruster_attempts = 0
        self.total_engine_failures = 0

In [15]:
#Create the Environments
# Original Environment

original_env = gym.make(
    "LunarLander-v3"
)

# Modified Environment

modified_env = ModifiedLunarLanderWrapper(

    gym.make("LunarLander-v3"),

    failure_probability=ENGINE_FAILURE_PROB,

    fuel_penalty=FUEL_PENALTY,

    landing_bonus=LANDING_BONUS
)

In [16]:
#Verify Action & Observation Spaces
print("Original Action Space:")
print(original_env.action_space)

print()

print("Modified Action Space:")
print(modified_env.action_space)

print()

print("Original Observation Space:")
print(original_env.observation_space)

print()

print("Modified Observation Space:")
print(modified_env.observation_space)

Original Action Space:
Discrete(4)

Modified Action Space:
Discrete(4)

Original Observation Space:
Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)

Modified Observation Space:
Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)


In [17]:
#Quick Smoke Test
state, info = modified_env.reset(seed=SEED)

print("Initial State Shape :", state.shape)

done = False

for i in range(10):

    action = modified_env.action_space.sample()

    next_state, reward, terminated, truncated, info = \
        modified_env.step(action)

    print(
        f"Step={i:2d} "
        f"Action={action} "
        f"Reward={reward:.2f}"
    )

    if terminated or truncated:

        break

Initial State Shape : (8,)
Step= 0 Action=1 Reward=1.78
Step= 1 Action=0 Reward=1.47
Step= 2 Action=2 Reward=-3.57
Step= 3 Action=1 Reward=1.74
Step= 4 Action=2 Reward=-3.34
Step= 5 Action=1 Reward=1.98
Step= 6 Action=2 Reward=-2.25
Step= 7 Action=2 Reward=0.96
Step= 8 Action=1 Reward=1.38
Step= 9 Action=2 Reward=-2.45


Part A2

In [ ]:
#Enhanced Wrapper for Verification
def step(self, action):

    # Store original action
    selected_action = action
    executed_action = selected_action

    # Simulate engine failure
    if selected_action in [1, 2, 3]:

        self.total_thruster_attempts += 1

        r = np.random.rand()

        if r < self.failure_probability:
            executed_action = 0
            self.total_engine_failures += 1

    # Execute action
    observation, base_reward, terminated, truncated, info = \
        self.env.step(executed_action)

    reward = base_reward

    ####################################################
    # Fuel Penalty
    ####################################################

    fuel_penalty_applied = False

    if selected_action in [1, 2, 3]:
        reward -= self.fuel_penalty
        fuel_penalty_applied = True

    ####################################################
    # Landing Bonus
    ####################################################

    landing_bonus_given = False

    safe_landing = (
        terminated
        and not truncated
        and observation[6] == 1
        and observation[7] == 1
        and abs(observation[2]) < 0.10
        and abs(observation[3]) < 0.10
        and abs(observation[4]) < 0.10
    )

    if safe_landing:
        reward += self.landing_bonus
        landing_bonus_given = True

    ####################################################
    # Store information for verification
    ####################################################

    self.last_selected_action = selected_action
    self.last_executed_action = executed_action
    self.last_base_reward = base_reward
    self.last_final_reward = reward
    self.last_penalty = fuel_penalty_applied
    self.last_bonus = landing_bonus_given

    return observation, reward, terminated, truncated, info

In [19]:
# Verification Function
def verify_environment(
    env,
    episodes=100
):

    env.reset_statistics()

    verification_log = []

    total_steps = 0

    total_thruster_attempts = 0

    total_failures = 0

    total_penalties = 0

    total_bonus = 0

    for episode in range(episodes):

        state, info = env.reset(seed=episode)

        terminated = False
        truncated = False

        while not (terminated or truncated):

            action = env.action_space.sample()

            next_state, reward, terminated, truncated, info = \
                env.step(action)

            verification_log.append({

                "Selected Action":
                env.last_selected_action,

                "Executed Action":
                env.last_executed_action,

                "Base Reward":
                env.last_base_reward,

                "Final Reward":
                env.last_final_reward,

                "Penalty":
                env.last_penalty,

                "Landing Bonus":
                env.last_bonus

            })

            total_steps += 1

            if env.last_selected_action in [1,2,3]:

                total_thruster_attempts += 1

            if (
                env.last_selected_action in [1,2,3]
                and env.last_executed_action == 0
            ):

                total_failures += 1

            if env.last_penalty:

                total_penalties += 1

            if env.last_bonus:

                total_bonus += 1

    failure_rate = (
        total_failures /
        total_thruster_attempts
    )

    print("="*60)

    print("Verification Summary")

    print("="*60)

    print(f"Episodes                  : {episodes}")

    print(f"Total Steps               : {total_steps}")

    print(f"Thruster Attempts         : {total_thruster_attempts}")

    print(f"Engine Failures           : {total_failures}")

    print(f"Failure Rate              : {failure_rate*100:.2f}%")

    print(f"Fuel Penalties Applied    : {total_penalties}")

    print(f"Landing Bonuses Awarded   : {total_bonus}")

    return pd.DataFrame(verification_log)

In [20]:
#Execute Verification
verification_df = verify_environment(

    modified_env,

    episodes=100

)

AttributeError: 'ModifiedLunarLanderWrapper' object has no attribute 'last_selected_action'

In [ ]:
#Display Verification Logs
verification_df.head(20)

In [ ]:
#Verify Requirement 1
failure_rate = (

    verification_df[
        (verification_df["Selected Action"] != 0)
        &
        (verification_df["Executed Action"] == 0)
    ].shape[0]

    /

    verification_df[
        verification_df["Selected Action"] != 0
    ].shape[0]

)

print(

    f"Observed Engine Failure Rate = {failure_rate*100:.2f}%"

)

In [ ]:
#Verify Requirement 2
thruster_attempts = verification_df[
    verification_df["Selected Action"] != 0
]

penalty_count = thruster_attempts[
    thruster_attempts["Penalty"] == True
].shape[0]

print("Thruster Attempts :", len(thruster_attempts))

print("Fuel Penalties :", penalty_count)

In [ ]:
#Verify Requirement 3
bonus_rows = verification_df[
    verification_df["Landing Bonus"] == True
]

print(

    "Successful Safe Landings :", len(bonus_rows)

)

bonus_rows.head()

PART A3

In [ ]:
#Plot 1: Engine Failure Statistics
# Calculate statistics
thruster_attempts = verification_df[
    verification_df["Selected Action"] != 0
].shape[0]

engine_failures = verification_df[
    (verification_df["Selected Action"] != 0) &
    (verification_df["Executed Action"] == 0)
].shape[0]

successful_fires = thruster_attempts - engine_failures

# Plot
plt.figure(figsize=(6,5))

plt.bar(
    ["Successful Thruster", "Engine Failure"],
    [successful_fires, engine_failures],
    color=["steelblue", "tomato"]
)

plt.title("Thruster Execution Statistics")
plt.ylabel("Count")

plt.grid(axis='y', alpha=0.3)

plt.show()

print(f"Failure Rate = {(engine_failures/thruster_attempts)*100:.2f}%")

In [ ]:
#Plot 2: Fuel Penalty Verification
penalty = verification_df["Penalty"].value_counts()

plt.figure(figsize=(6,5))

plt.bar(
    penalty.index.astype(str),
    penalty.values,
    color=["green","orange"]
)

plt.title("Fuel Penalty Application")

plt.xlabel("Fuel Penalty Applied")

plt.ylabel("Count")

plt.grid(axis='y', alpha=0.3)

plt.show()

print("Penalty Statistics")
print(penalty)

In [ ]:
#Plot 3: Selected vs Executed Actions
selected = verification_df["Selected Action"].value_counts().sort_index()

executed = verification_df["Executed Action"].value_counts().sort_index()

x = np.arange(4)

width = 0.35

plt.figure(figsize=(8,5))

plt.bar(
    x-width/2,
    selected.values,
    width,
    label="Selected"
)

plt.bar(
    x+width/2,
    executed.values,
    width,
    label="Executed"
)

plt.xticks(x,[0,1,2,3])

plt.xlabel("Action")

plt.ylabel("Frequency")

plt.title("Selected vs Executed Actions")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

In [ ]:
#Plot 4: Landing Bonus
bonus = verification_df["Landing Bonus"].value_counts()

plt.figure(figsize=(5,5))

plt.bar(
    bonus.index.astype(str),
    bonus.values,
    color=["purple","gold"]
)

plt.title("Landing Bonus Awarded")

plt.ylabel("Count")

plt.grid(axis='y', alpha=0.3)

plt.show()

In [ ]:
#Summary Table
summary = {

    "Metric":[

        "Thruster Attempts",

        "Engine Failures",

        "Failure Rate (%)",

        "Fuel Penalties",

        "Landing Bonuses"

    ],

    "Value":[

        thruster_attempts,

        engine_failures,

        round(engine_failures/thruster_attempts*100,2),

        verification_df["Penalty"].sum(),

        verification_df["Landing Bonus"].sum()

    ]

}

summary_df = pd.DataFrame(summary)

summary_df

##Assignment Report: Verification Results

### Requirement 1

Approximately 15% of all attempted thruster actions were replaced
with the Do Nothing action.

The observed failure rate was approximately 15%, validating the
correct implementation of stochastic engine failures.

---

### Requirement 2

Fuel penalty was applied for every attempted thruster action,
regardless of whether the engine fired successfully or was replaced
by Do Nothing.

The number of penalties exactly matched the number of attempted
thruster actions.

---

### Requirement 3

The +50 landing bonus was awarded only when the strict safe
landing conditions were satisfied.

Under a random policy, safe landings were extremely rare, therefore
very few (or zero) bonus rewards were observed, which is expected.